# Phase 10 plan 10-13: GRU + transformer sequence candidates (Colab)

Trains the two GPU-bound D-12 sequence candidates on the bundle exported by
`models.bracket.registry.export_sequence_bundle`. Splits are read from
`manifest.json` -- see `colab/README.md`'s "Split contract": this notebook
must never derive its own train/val/test boundaries.

Upload this notebook plus the exported `colab_input/` directory (containing
`manifest.json`, `<season>_tensors.npz`, `<season>_ids.parquet` per season)
before running. Select a GPU runtime (Runtime -> Change runtime type -> GPU)
before Cell 1.


In [ ]:
# Cell 1: pinned installs, matching this project's own lockfiles
# (requirements.txt / requirements-rl.txt) so the notebook's torch/numpy/
# pandas/pyarrow versions match the local harness that later scores these
# predictions.
!pip install -q torch==2.12.0 numpy==2.5.2 pandas==3.0.5 pyarrow==25.0.1

import numpy
import pandas
import pyarrow
import torch

print("torch", torch.__version__)
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("pyarrow", pyarrow.__version__)
print("cuda available:", torch.cuda.is_available())


In [ ]:
# Cell 2: load the exported bundle + manifest.json, verify the feature
# vintage this bundle was built from before touching any tensor.
import json
from pathlib import Path

# Upload colab_input/ next to this notebook (Colab's file browser, left
# sidebar), or mount Drive and point BUNDLE_DIR at the uploaded copy there.
BUNDLE_DIR = Path("colab_input")

manifest = json.loads((BUNDLE_DIR / "manifest.json").read_text())
print("seq_window:", manifest["seq_window"])
print("n_seq_stats:", len(manifest["seq_stats"]))
print("n_static_cols:", len(manifest["static_cols"]))
print("seasons_exported:", manifest["seasons_exported"])
print("test seasons in this bundle:", sorted(manifest["splits"]))

# The manifest records the sha256 of the LOCAL features.parquet this bundle
# was built from -- print it for the operator to cross-check against the
# downloader's own recorded value; this notebook has no way to independently
# recompute it (features.parquet itself is never uploaded, only its
# derived tensors), so this is a print-and-eyeball check, not an assertion.
print("features_sha256 (cross-check against the export log):",
      manifest["features_sha256"])


In [ ]:
# Cell 3: the two candidate model classes, copied VERBATIM from
# models/bracket/recurrent.py and models/bracket/transformer.py.
#
# THESE ARE COPIES, NOT IMPORTS -- this notebook has no access to the
# project's own Python package, so the class bodies are duplicated here by
# hand. A divergence between this cell and the real recurrent.py/
# transformer.py invalidates the comparison this whole handoff exists to
# make: keep the two in lockstep by eye on every edit. manifest.json's
# `features_sha256` binds a given run to one input vintage, but nothing
# binds this cell to the repo's own model code except discipline.
from torch import nn


class GruSeqRegressor(nn.Module):
    """GRU (not LSTM, per D-12): fewer parameters than an LSTM for the
    same window, which matters against D-13's fixed compute budget, and the
    10-gameweek window is short enough that LSTM's extra gating buys little.
    Selects the LAST NON-PADDED hidden state via the mask, never the last
    index blindly; concatenates the static side-vector AFTER the GRU.
    """

    def __init__(self, n_stats, n_static, hidden_size=32, num_layers=1, dropout=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.gru = nn.GRU(input_size=n_stats, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(hidden_size + n_static, 32), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x_seq, mask, x_static):
        x = x_seq.masked_fill(mask.unsqueeze(-1), 0.0)
        out, _ = self.gru(x)
        lengths = (~mask).sum(dim=1)
        batch_idx = torch.arange(x.shape[0], device=x.device)
        last_idx = (lengths - 1).clamp(min=0)
        selected = out[batch_idx, last_idx]
        zero_len = (lengths == 0).unsqueeze(1)
        selected = torch.where(zero_len, torch.zeros_like(selected), selected)
        combined = torch.cat([selected, x_static], dim=1)
        return self.head(combined).squeeze(-1)


# 10-01-PLAN.md's locked "Transformer budget" row.
D_MODEL = 64
N_HEAD = 4
N_LAYERS = 2
DIM_FF = 128
DROPOUT = 0.1


class TransformerSeqRegressor(nn.Module):
    """Locked size budget (D_MODEL/N_HEAD/N_LAYERS/DIM_FF above). Mean-
    pools the encoder output over non-padded positions only, concatenating
    the static side-vector AFTER pooling."""

    def __init__(self, n_stats, n_static, window, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(n_stats, D_MODEL)
        self.pos_embed = nn.Parameter(torch.zeros(window, D_MODEL))
        nn.init.normal_(self.pos_embed, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=D_MODEL, nhead=N_HEAD,
                                           dim_feedforward=DIM_FF, dropout=dropout,
                                           batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head = nn.Sequential(
            nn.Linear(D_MODEL + n_static, 32), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(32, 1))

    def forward(self, x_seq, mask, x_static):
        x = x_seq.masked_fill(mask.unsqueeze(-1), 0.0)
        h = self.input_proj(x) + self.pos_embed.unsqueeze(0)
        enc = self.encoder(h, src_key_padding_mask=mask)
        enc = enc.masked_fill(mask.unsqueeze(-1), 0.0)
        lengths = (~mask).sum(dim=1)
        denom = lengths.clamp(min=1).unsqueeze(1).to(enc.dtype)
        pooled = enc.sum(dim=1) / denom
        zero_len = (lengths == 0).unsqueeze(1)
        pooled = torch.where(zero_len, torch.zeros_like(pooled), pooled)
        combined = torch.cat([pooled, x_static], dim=1)
        return self.head(combined).squeeze(-1)


In [ ]:
# Cell 4: pin the seed from the manifest (falls back to 0 if the manifest
# carries none -- this bundle's manifest does not currently record one, so
# this is the single place a fixed seed enters the run).
import numpy as np

SEED = manifest.get("seed", 0)
torch.manual_seed(SEED)
np.random.seed(SEED)
print("seed:", SEED)


In [ ]:
# Cell 5: the per-test-season loop -- reads manifest["splits"] for the
# train/val/test triple (the split authority, per colab/README.md's "Split
# contract" -- never derive a boundary here), trains both candidates,
# predicts on the test season, and writes one parquet per candidate per
# test season with EXACTLY the six contract columns
# (season, gw, player_code, fixture_id, xp_med, xp_mean).
import time


def load_season(season):
    npz = np.load(BUNDLE_DIR / f"{season}_tensors.npz")
    ids = pandas.read_parquet(BUNDLE_DIR / f"{season}_ids.parquet")
    # NaN-scrub x_seq/x_static exactly as the local adapter does
    # (models/bracket/recurrent.py::_SeqRegressorAdapter._batches wraps both
    # in torch.nan_to_num): the exported tensors legitimately carry NaN in
    # enrichment-family columns for early seasons, and an unscrubbed NaN
    # poisons the loss, which poisons early stopping.
    return {
        "x_seq": torch.nan_to_num(torch.as_tensor(npz["x_seq"], dtype=torch.float32)),
        "mask": torch.as_tensor(npz["mask"], dtype=torch.bool),
        "x_static": torch.nan_to_num(torch.as_tensor(npz["x_static"], dtype=torch.float32)),
        "y": torch.as_tensor(npz["y"], dtype=torch.float32),
        "ids": ids,
    }


def concat_seasons(seasons):
    parts = [load_season(s) for s in seasons]
    return {
        "x_seq": torch.cat([p["x_seq"] for p in parts], dim=0),
        "mask": torch.cat([p["mask"] for p in parts], dim=0),
        "x_static": torch.cat([p["x_static"] for p in parts], dim=0),
        "y": torch.cat([p["y"] for p in parts], dim=0),
        "ids": pandas.concat([p["ids"] for p in parts], ignore_index=True),
    }


def train_one(model_cls, model_kwargs, train_bundle, val_bundle, *,
             max_epochs=60, patience=10, lr=1e-3, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model_cls(n_stats=train_bundle["x_seq"].shape[-1],
                      n_static=train_bundle["x_static"].shape[-1],
                      **model_kwargs).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()

    xs, m, xst, y = (t.to(device) for t in (
        train_bundle["x_seq"], train_bundle["mask"], train_bundle["x_static"],
        train_bundle["y"]))
    vxs, vm, vxst, vy = (t.to(device) for t in (
        val_bundle["x_seq"], val_bundle["mask"], val_bundle["x_static"],
        val_bundle["y"]))

    # MINI-BATCH training, matching the local adapter's loop shape. Full-batch
    # was not just a fidelity gap: with dropout active, torch's efficient
    # attention kernel cannot produce dropout seeds for a batch > 65,535, so
    # the transformer candidate crashes outright on the ~200K-row 8-season
    # concatenated train set. 8192 (vs the local CPU loop's 256) keeps GPU
    # throughput without approaching the kernel limit.
    batch_size = 8192
    n = xs.shape[0]
    # Initialise best_state from the INITIAL weights (never None), matching
    # the local loop -- if every epoch's val_loss is non-finite, we still
    # restore a valid state_dict instead of crashing in load_state_dict.
    best_val, best_epoch = float("inf"), -1
    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    gen = torch.Generator().manual_seed(0)
    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n, generator=gen)
        for start in range(0, n, batch_size):
            b = perm[start:start + batch_size].to(device)
            opt.zero_grad()
            loss = loss_fn(model(xs[b], m[b], xst[b]), y[b])
            loss.backward()
            opt.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for start in range(0, vxs.shape[0], batch_size):
                sl = slice(start, start + batch_size)
                val_losses.append(loss_fn(model(vxs[sl], vm[sl], vxst[sl]), vy[sl]).item())
        val_loss = sum(val_losses) / max(len(val_losses), 1)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        elif epoch - best_epoch >= patience:
            break
    model.load_state_dict(best_state)
    return model, device


CANDIDATES = {
    "rnn": (GruSeqRegressor, {"hidden_size": 32, "num_layers": 1, "dropout": 0.1}),
    "transformer": (TransformerSeqRegressor, {"window": manifest["seq_window"],
                                              "dropout": DROPOUT}),
}

OUT_DIR = Path("colab_predictions")
OUT_DIR.mkdir(exist_ok=True)

for candidate, (model_cls, model_kwargs) in CANDIDATES.items():
    for test_season, split in manifest["splits"].items():
        print(f"[{candidate}] {test_season}: train={split['train_seasons']} "
              f"val={split['val_season']}")
        t0 = time.monotonic()

        train_bundle = concat_seasons(split["train_seasons"])
        val_bundle = load_season(split["val_season"])
        test_bundle = load_season(test_season)

        model, device = train_one(model_cls, model_kwargs, train_bundle, val_bundle)

        model.eval()
        preds = []
        with torch.no_grad():
            txs = test_bundle["x_seq"].to(device)
            tm = test_bundle["mask"].to(device)
            txst = test_bundle["x_static"].to(device)
            for start in range(0, txs.shape[0], 8192):
                sl = slice(start, start + 8192)
                preds.append(model(txs[sl], tm[sl], txst[sl]).cpu())
        xp_med = torch.cat(preds).numpy()

        out = test_bundle["ids"][["season", "gw", "player_code", "fixture_id"]].copy()
        out["xp_med"] = xp_med
        out["xp_mean"] = xp_med   # single-objective candidate: med == mean
        out = out[["season", "gw", "player_code", "fixture_id", "xp_med", "xp_mean"]]

        dest = OUT_DIR / f"colab_{candidate}_{test_season}.parquet"
        out.to_parquet(dest, index=False)
        wall = round(time.monotonic() - t0, 1)
        print(f"[{candidate}] {test_season}: wrote {dest} ({len(out)} rows, {wall}s)")


In [ ]:
# Cell 6: per-candidate wall clock is printed above by Cell 5. Before
# downloading colab_predictions/*.parquet, open Colab's usage panel
# (top-right corner) and RECORD the compute units consumed and remaining --
# per colab/README.md's "Compute accounting": D-13's budget is all 200
# owned units with a hard stop at exhaustion, and an unrecorded spend cannot
# be hard-stopped. Paste the wall clock, units consumed/remaining, and the
# downloaded filenames into the checkpoint response.
print("Remember: record Colab compute units consumed/remaining BEFORE "
      "downloading colab_predictions/*.parquet.")
